# FB Marketplace Deal Watcher (Notebook version)

Scrapes Facebook Marketplace searches for computer parts and flags listings priced
well below what you've typically seen for that search term.

**Run the cells top to bottom, in order.** A few things specific to running this
in a notebook instead of as scripts:

- Playwright's *async* API is used here (not the sync API from the script version),
  because Jupyter already runs its own event loop and the sync API can't run
  inside one. Cells use `await` directly, which works out of the box in Jupyter.
- Login (Section 2) is split across two cells so you can log into Facebook by hand
  in the window that opens, then come back and run the next cell to save the session.
- The continuous watcher (Section 6) runs an infinite loop and will **block this
  kernel** for as long as it runs. Use the Jupyter "Interrupt Kernel" button (■) to
  stop it. Section 5 gives you a single non-blocking test cycle first.

**Also worth knowing:** automated browsing of Marketplace is against Facebook's
Terms of Service. This is fine for a personal tool used at a reasonable pace, but
keep the interval slow, expect occasional login checkpoints, and don't run this at
scale or share the scraped session state.

## 0. Install dependencies

Run once per environment.

In [ ]:
import sys

%pip install -r requirements.txt
!{sys.executable} -m playwright install chromium

Note: you may need to restart the kernel to use updated packages.


## 1. Config

Edit `SEARCHES` and `LOCATION` in `config.py`; this cell imports the settings from that file.

In [8]:
import re
import os
import json
import random
import sys
import asyncio
import sqlite3
import logging
from statistics import median
from urllib.parse import quote
from contextlib import contextmanager

from playwright.async_api import async_playwright
from config import (
    LOCATION,
    RADIUS_MILES,
    SEARCHES,
    MIN_HISTORY_FOR_DEAL_CHECK,
    POLL_INTERVAL_MIN_MINUTES,
    POLL_INTERVAL_MAX_MINUTES,
    AUTH_STATE_PATH,
    DB_PATH,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("fb-watcher")

## 2. Log into Facebook (run once, or again whenever the session expires)

Run the next cell: a real browser window opens. Log into Facebook in it
(handle any 2FA/checkpoint), then come back and run the cell **after** it to
save the session.

In [6]:
import threading

class BackgroundLoop:
    def __init__(self):
        if sys.platform == "win32":
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        self.loop = asyncio.new_event_loop()
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.thread.start()

    def _run(self):
        asyncio.set_event_loop(self.loop)
        self.loop.run_forever()

    def run(self, coro):
        """Run an async coroutine on the background loop and block until it's done."""
        future = asyncio.run_coroutine_threadsafe(coro, self.loop)
        return future.result()

bg = BackgroundLoop()
print("Background event loop started.")

Background event loop started.


In [7]:
_pw = await async_playwright().start()
_login_browser = await _pw.chromium.launch(headless=False)
_login_context = await _login_browser.new_context()
_login_page = await _login_context.new_page()
await _login_page.goto("https://www.facebook.com/login")
print("Log into Facebook in the window that just opened, then run the next cell.")

NotImplementedError: 

In [ ]:
await _login_context.storage_state(path=AUTH_STATE_PATH)
await _login_browser.close()
await _pw.stop()
print(f"Session saved to {AUTH_STATE_PATH}")

## 3. Storage (SQLite: tracks seen listings + price history)

In [ ]:
SCHEMA = '''
CREATE TABLE IF NOT EXISTS listings (
    listing_id TEXT PRIMARY KEY,
    search_term TEXT NOT NULL,
    title TEXT,
    price INTEGER,
    url TEXT,
    location TEXT,
    first_seen TEXT DEFAULT CURRENT_TIMESTAMP
);
'''

@contextmanager
def get_conn():
    conn = sqlite3.connect(DB_PATH)
    try:
        yield conn
    finally:
        conn.close()

def init_db():
    with get_conn() as conn:
        conn.execute(SCHEMA)
        conn.commit()

def is_new(listing_id):
    with get_conn() as conn:
        row = conn.execute("SELECT 1 FROM listings WHERE listing_id = ?", (listing_id,)).fetchone()
        return row is None

def price_history(search_term):
    with get_conn() as conn:
        rows = conn.execute(
            "SELECT price FROM listings WHERE search_term = ? AND price IS NOT NULL",
            (search_term,),
        ).fetchall()
        return [r[0] for r in rows]

def median_price(search_term):
    hist = price_history(search_term)
    return median(hist) if hist else None

def save_listing(listing_id, search_term, title, price, url, location):
    with get_conn() as conn:
        conn.execute(
            '''INSERT OR IGNORE INTO listings
               (listing_id, search_term, title, price, url, location)
               VALUES (?, ?, ?, ?, ?, ?)''',
            (listing_id, search_term, title, price, url, location),
        )
        conn.commit()

init_db()
print("DB ready at", DB_PATH)

## 4. Scraper

Relies on `/marketplace/item/<id>/` links and each card's visible text rather
than CSS classes (Facebook's class names are auto-generated and change often).
If results come back empty or garbled, Facebook likely changed the layout —
set `headless=False` in a test call below to watch it load, or run
`playwright codegen https://www.facebook.com/marketplace` in a terminal to see
the current structure and adjust `extract_listings` accordingly.

In [ ]:
ITEM_URL_RE = re.compile(r"/marketplace/item/(\d+)")

def build_search_url(term):
    return (
        f"https://www.facebook.com/marketplace/{LOCATION}/search"
        f"?query={quote(term)}&radius={RADIUS_MILES}&exact=false"
    )

async def extract_listings(page, search_term):
    for _ in range(4):
        await page.mouse.wheel(0, 2000)
        await page.wait_for_timeout(800)

    anchors = await page.locator("a[href*='/marketplace/item/']").all()
    results = []
    seen_ids = set()

    for a in anchors:
        href = await a.get_attribute("href") or ""
        m = ITEM_URL_RE.search(href)
        if not m:
            continue
        listing_id = m.group(1)
        if listing_id in seen_ids:
            continue
        seen_ids.add(listing_id)

        text = (await a.inner_text()).strip()
        lines = [l.strip() for l in text.split("\n") if l.strip()]

        price = None
        for line in lines:
            price_match = re.search(r"\$[\d,]+", line)
            if price_match and price is None:
                price = int(price_match.group(0).replace("$", "").replace(",", ""))

        non_price_lines = [l for l in lines if not re.fullmatch(r"\$[\d,]+", l)]
        title = max(non_price_lines, key=len) if non_price_lines else None
        location = lines[-1] if len(lines) >= 2 else ""

        results.append({
            "listing_id": listing_id,
            "title": title or "(unknown title)",
            "price": price,
            "url": f"https://www.facebook.com/marketplace/item/{listing_id}/",
            "location": location,
            "search_term": search_term,
        })

    return results

async def scrape_search(term, max_price, headless=True):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=headless)
        context = await browser.new_context(storage_state=AUTH_STATE_PATH)
        page = await context.new_page()

        url = build_search_url(term)
        logger.info(f"Scraping: {url}")
        await page.goto(url, wait_until="domcontentloaded")
        await page.wait_for_timeout(3000)

        if "login" in page.url:
            logger.error("Redirected to login — session expired. Redo Section 2.")
            await browser.close()
            return []

        listings = await extract_listings(page, term)
        await browser.close()

    return [l for l in listings if l["price"] is None or l["price"] <= max_price]

def notify(title, message):
    logger.info(f"NOTIFY: {title} | {message}")
    try:
        from plyer import notification
        notification.notify(title=title, message=message, timeout=15)
    except Exception as e:
        logger.warning(f"Desktop notification failed ({e}); logged instead.")

## 5. One test cycle (non-blocking)

Runs every search in `SEARCHES` once, saves new listings, and notifies on
anything that looks like a deal. Good for checking your config and selectors
work before starting the long-running watcher in Section 6.

In [ ]:
async def run_cycle():
    for search in SEARCHES:
        term = search["term"]
        max_price = search["max_price"]
        deal_ratio = search["deal_ratio"]

        try:
            listings = await scrape_search(term, max_price)
        except Exception as e:
            logger.exception(f"Scrape failed for '{term}': {e}")
            continue

        logger.info(f"'{term}': {len(listings)} listings under ${max_price}")

        for listing in listings:
            if not is_new(listing["listing_id"]):
                continue

            history = price_history(term)
            med = median_price(term)
            is_deal = (
                listing["price"] is not None
                and med is not None
                and len(history) >= MIN_HISTORY_FOR_DEAL_CHECK
                and listing["price"] <= deal_ratio * med
            )

            save_listing(
                listing["listing_id"], term, listing["title"],
                listing["price"], listing["url"], listing["location"],
            )

            price_str = f"${listing['price']}" if listing["price"] is not None else "?"
            if is_deal:
                notify(f"Deal: {term}", f"{listing['title']} — {price_str} (median ~${med:.0f})\n{listing['url']}")
                logger.info(f"DEAL FLAGGED: {listing['title']} {price_str} {listing['url']}")
            else:
                logger.info(f"New listing: {listing['title']} {price_str} {listing['url']}")

await run_cycle()

## 6. Continuous watcher (blocking)

Loops forever, sleeping a randomized 60-120 minutes between cycles (see
`POLL_INTERVAL_*` in Section 1). This **blocks the kernel** while it runs —
use Jupyter's Interrupt Kernel button to stop it. Only run this cell once
you've confirmed Section 5 finds real listings.

In [ ]:
async def run_forever():
    while True:
        logger.info("Starting scrape cycle")
        await run_cycle()
        wait_minutes = random.uniform(POLL_INTERVAL_MIN_MINUTES, POLL_INTERVAL_MAX_MINUTES)
        logger.info(f"Cycle done. Sleeping {wait_minutes:.1f} minutes.")
        await asyncio.sleep(wait_minutes * 60)

await run_forever()

## 7. Query what's been found

Quick way to check the database without a separate tool.

In [ ]:
with get_conn() as conn:
    rows = conn.execute(
        "SELECT search_term, title, price, url, first_seen FROM listings ORDER BY first_seen DESC LIMIT 20"
    ).fetchall()
for r in rows:
    print(r)